# Your first yats experiment

**What is yats?** yats is a self-hosted trading-research platform: market data
from three vendors lands in QuestDB, features are computed under strict
point-in-time discipline, and strategies are trained and judged with purged
walk-forward optimization under realistic execution. Every experiment is a
frozen, content-addressed spec, so any number the platform reports can be
traced back to the exact configuration that produced it. Between a good-looking
backtest and real orders stands a promotion pipeline whose statistical gates
exist to say "no".

**The honesty thesis.** Most trading ideas are noise, and a research platform
earns its keep by telling you so *before the market does*. Everything in this
notebook — the forced one-day execution lag, the purged gap in every
walk-forward fold, the deflated Sharpe ratio, a trial counter that only counts
up — is machinery for making it hard for a bad idea to look good. The
platform's own scoreboard is the proof: after 63 recorded trials across every
strategy family tried so far, **zero strategies have been certified** (closest
attempt: strict deflated-Sharpe probability 0.921 against a 0.95 bar — see
[`docs/research/full_span_verdicts.md`](research/full_span_verdicts.md)).
That is the system working, not failing.

This walkthrough assumes no prior yats knowledge. Every code cell degrades
gracefully if you don't have the database running, and the committed outputs
show what a live run looks like — you can just read.

## 1. The data model in one picture

Three layers, one direction of flow, no edits in place:

```text
 vendor APIs                       QuestDB
 ───────────                       ────────────────────────────────────────────
 Alpaca ───────────┐               raw_alpaca_equity_ohlcv        ┐ append-only,
 ThetaData ────────┼── ingest ───▶ raw_thetadata_options_eod      ├ vendor-shaped,
 financialdatasets ┘               raw_fd_fundamentals, …         ┘ never edited
                                          │
                                          │  canonicalize: reconcile + dedup
                                          ▼  (latest-ingested-wins, lineage cols)
                                   canonical_equity_ohlcv         ┐ point-in-time:
                                   canonical_options_chain        ├ a row is only
                                   canonical_fundamentals, …      ┘ visible as-of
                                          │                         its date
                                          │  feature_pipeline: incremental,
                                          ▼  per-symbol watermarks
                                   features   (sets: core_v1, options_v1,
                                               insider_v1)
```

- **Raw** tables are exactly what the vendor sent, append-only. Re-running an
  ingest is always safe: every row carries its run id and readers resolve
  duplicates.
- **Canonical** tables are the reconciled, point-in-time truth (e.g.
  `canonical_fundamentals` stores *point-in-time* fundamentals — what was
  knowable on that date, not the restated version published later).
- **Features** are computed incrementally from canonical data; watermarks track
  what has already been computed per symbol.

Experiments only ever read canonical and feature tables. That is the first
honesty mechanism: no code path lets a model see vendor revisions from the
future.

## 2. Load some data

QuestDB serves SQL over plain HTTP on `localhost:9000` — no driver needed.
If you have the stack running (`docker compose up -d`, then a backfill per
[`docs/ingestion.md`](ingestion.md)), the cells below query it live. If not,
they print:

```text
QuestDB not reachable — that's fine, keep reading.
Every data cell below degrades to a message like this one;
the committed outputs show what a live run looks like.
```

In [1]:
import json, urllib.parse, urllib.request

QDB = "http://localhost:9000/exec"   # QuestDB HTTP endpoint (docker compose up -d)

def qdb(sql: str):
    """Run SQL against QuestDB over HTTP; returns (rows, column_names)."""
    url = QDB + "?query=" + urllib.parse.quote(sql)
    with urllib.request.urlopen(url, timeout=5) as r:
        d = json.load(r)
    if "error" in d:
        raise RuntimeError(d["error"])
    return d.get("dataset", []), [c["name"] for c in d.get("columns", [])]

try:
    tables, _ = qdb("SELECT table_name FROM tables() ORDER BY table_name")
    names = [t[0] for t in tables]
    DB_UP = True
    print(f"QuestDB is up — {len(names)} tables.")
    layers = [n for n in names if n.startswith(("raw_", "canonical_", "feature"))]
    print("data-model tables:")
    for n in layers:
        print("  ", n)
except Exception:
    DB_UP = False
    print("QuestDB not reachable — that's fine, keep reading.")
    print("Every data cell below degrades to a message like this one;")
    print("the committed outputs show what a live run looks like.")

QuestDB is up — 33 tables.
data-model tables:
   canonical_equity_ohlcv
   canonical_financial_metrics
   canonical_fundamentals
   canonical_hashes
   canonical_insider_trades
   canonical_inst_ownership
   canonical_institutional_holdings
   canonical_options_chain
   canonical_pins
   feature_watermarks
   features
   raw_alpaca_equity_ohlcv
   raw_fd_analyst_estimates
   raw_fd_earnings
   raw_fd_financial_metrics
   raw_fd_fundamentals
   raw_fd_insider_trades
   raw_fd_institutional_holdings
   raw_thetadata_options_chain
   raw_thetadata_options_eod


In [2]:
if DB_UP:
    rows, _ = qdb(
        "SELECT symbol, count() AS bars, min(timestamp) AS first_bar, "
        "max(timestamp) AS last_bar FROM canonical_equity_ohlcv "
        "WHERE symbol IN ('AAPL','MSFT','NVDA') GROUP BY symbol ORDER BY symbol"
    )
    print(f"{'symbol':<8}{'bars':>6}  {'first_bar':<22}{'last_bar'}")
    for sym, bars, first, last in rows:
        print(f"{sym:<8}{bars:>6}  {first[:10]:<22}{last[:10]}")
else:
    print("Skipped — no database. A live run prints per-symbol bar coverage here.")

symbol    bars  first_bar             last_bar
AAPL      1633  2020-01-02            2026-07-02
MSFT      1633  2020-01-02            2026-07-02
NVDA      1633  2020-01-02            2026-07-02


In [3]:
if DB_UP:
    rows, cols = qdb(
        "SELECT timestamp, symbol, open, close, volume "
        "FROM canonical_equity_ohlcv WHERE symbol = 'AAPL' "
        "ORDER BY timestamp DESC LIMIT 5"
    )
    print(" | ".join(cols))
    for r in rows:
        print(r[0][:10], "|", r[1], "|", r[2], "|", r[3], "|", r[4])
else:
    print("Skipped — no database. A live run prints the latest AAPL bars here.")

timestamp | symbol | open | close | volume
2026-07-02 | AAPL | 294.12 | 308.63 | 75580907
2026-07-01 | AAPL | 293.44 | 294.38 | 50332267
2026-06-30 | AAPL | 281.17 | 289.36 | 65405945
2026-06-29 | AAPL | 286.73 | 281.74 | 66820039
2026-06-26 | AAPL | 275.0 | 283.78 | 261946552


## 3. Define an experiment — `ExperimentSpec`

An experiment is a frozen dataclass (`research/experiments/spec.py`). Its
`experiment_id` is the SHA-256 of its canonical JSON — same config, same id,
forever. The required fields:

| field | meaning |
|---|---|
| `experiment_name` | human label |
| `symbols` | tuple of tickers (sorted + deduped for you) |
| `start_date` / `end_date` | evaluation span |
| `interval` | `"daily"` (the only supported bar size) |
| `feature_set` | which feature YAML to use, e.g. `core_v1` |
| `policy` | `equal_weight`, `sma`, `ppo`, `sac_*`, or `hierarchical` |
| `policy_params` | policy hyperparameters (incl. `reward_version`) |
| `cost_config` | transaction costs in basis points |
| `seed` | RNG seed |

Optional fields add walk-forward config, risk overlays, regime awareness — and
two small fields that carry most of the honesty weight:

- **`execution_lag_days` (default `1`)** — you decide at today's close, you
  fill *tomorrow*. Features are end-of-day aggregates: today's close is part of
  the very signal you are trading on, so a same-bar fill means transacting at a
  price that did not exist when the decision was made. That is free lookahead,
  and it silently inflates every backtest metric.
- **`fill_timing` (default `"next_close"`)** — *when* tomorrow you fill:
  `next_close` (fill at close(t+1)) or `next_open` (fill at open(t+1)).
  Same-bar fills survive only as a legacy `execution_lag_days=0` escape hatch
  that warns loudly and exists to reproduce old results, not to create new ones.

In [4]:
import sys
from datetime import date
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # run this notebook from the repo root

try:
    from research.experiments.spec import ExperimentSpec, CostConfig
    HAVE_YATS = True
except Exception as e:
    HAVE_YATS = False
    print("Could not import yats — run from the repo root after `uv sync`.")
    print("Reason:", e)

if HAVE_YATS:
    spec = ExperimentSpec(
        experiment_name="first_experiment_demo",
        symbols=("AAPL", "MSFT", "NVDA"),
        start_date=date(2020, 1, 2),
        end_date=date(2026, 7, 2),
        interval="daily",
        feature_set="core_v1",
        policy="ppo",
        policy_params={"total_timesteps": 2000, "reward_version": "v1"},
        cost_config=CostConfig(transaction_cost_bp=5.0),
        seed=7,
    )
    print("policy:            ", spec.policy)
    print("symbols:           ", spec.symbols)
    print("execution_lag_days:", spec.execution_lag_days, "(decide at today's close ...)")
    print("fill_timing:       ", repr(spec.fill_timing), "(... fill at tomorrow's close)")

policy:             ppo
symbols:            ('AAPL', 'MSFT', 'NVDA')
execution_lag_days: 1 (decide at today's close ...)
fill_timing:        'next_close' (... fill at tomorrow's close)


The spec doesn't just default to honest execution — it refuses dishonest
combinations outright:

In [5]:
if HAVE_YATS:
    for kwargs in (
        {"fill_timing": "same_close"},                          # not even a valid value
        {"fill_timing": "next_open", "execution_lag_days": 0},  # inconsistent combo
    ):
        try:
            ExperimentSpec(
                experiment_name="cheater", symbols=("AAPL",),
                start_date=date(2020, 1, 2), end_date=date(2026, 7, 2),
                interval="daily", feature_set="core_v1", policy="ppo",
                policy_params={}, cost_config=CostConfig(transaction_cost_bp=5.0),
                seed=1, **kwargs,
            )
        except ValueError as e:
            print("rejected ->", e)
else:
    print("Skipped — spec import unavailable (see previous cell).")

rejected -> fill_timing must be 'next_close' or 'next_open', got 'same_close'
rejected -> fill_timing='next_open' requires execution_lag_days=1; same-day fills exist only under legacy execution_lag_days=0 with fill_timing='next_close'


## 4. One walk-forward fold, on paper

A single train/test split is a lottery ticket: your result is hostage to one
particular stretch of market. yats evaluates with **anchored walk-forward
optimization (WFO)**: train on everything up to a point, test on the next
block, extend the training window, repeat. Each fold trains its own model to
full convergence, and the out-of-sample (OOS) blocks are concatenated into one
track record.

Between every train window and its test block sits a **purged gap**, because
two things leak across the boundary:

1. **Labels** — a model predicting 21-day forward returns has training labels
   near the boundary that overlap the test period. Those bars are purged
   (`label_horizon`).
2. **Features** — rolling features (e.g. a 63-day average) computed early in
   the test block still contain training-period prices. A buffer sized from the
   feature set's *maximum lookback* covers this (`purge_buffer`).

```text
 bars ────────────────────────────────────────────────────────▶ time
 [ train ................................ ][ purged gap ][ test (OOS) ]
   fit the model here                        84 bars       42 bars the
   (final 21 bars lose their labels —        nobody may    model has
    the labels peek into the gap)            touch         never seen
```

The cell below re-derives the fold table from
`research/eval/wfo.py::build_wfo_folds` arithmetic using the real geometry of
the ALPHA-1 supervised sweep
([`docs/research/wfo_sweep_alpha1_results.md`](research/wfo_sweep_alpha1_results.md)):
502 trading days, 4 folds, 21-day labels, 63-bar feature lookback.

In [6]:
# Pure-stdlib re-derivation of research/eval/wfo.py::build_wfo_folds (anchored mode)
# using the real ALPHA-1 sweep geometry (docs/research/wfo_sweep_alpha1_results.md).

n_bars, n_folds = 502, 4          # dev10 universe, 2024-07-01..2026-07-02
train_window    = 250             # fold-1 anchor
label_horizon   = 21              # bars purged: the 21-day label leaks past train-end
purge_buffer    = 63              # auto-sized from the feature set's max lookback

gap = label_horizon + purge_buffer                       # 84 bars
test_window = (n_bars - train_window - gap) // n_folds   # 42 bars per fold

print(f"purge gap = {label_horizon} + {purge_buffer} = {gap} bars; "
      f"test window = {test_window} bars\n")
print(f"{'fold':<6}{'train (effective)':<22}{'purged gap':<16}{'test (OOS)'}")
for i in range(n_folds):
    raw_train_end = train_window + i * test_window       # anchored: train grows
    eff_train_end = raw_train_end - gap                  # purge before test
    test_start, test_end = raw_train_end, raw_train_end + test_window
    print(f"{i+1:<6}[0, {eff_train_end})".ljust(28)
          + f"[{eff_train_end}, {test_start})".ljust(16)
          + f"[{test_start}, {test_end})")
print(f"\ntotal OOS bars: {n_folds * test_window}")

purge gap = 21 + 63 = 84 bars; test window = 42 bars

fold  train (effective)     purged gap      test (OOS)
1     [0, 166)              [166, 250)      [250, 292)
2     [0, 208)              [208, 292)      [292, 334)
3     [0, 250)              [250, 334)      [334, 376)
4     [0, 292)              [292, 376)      [376, 418)

total OOS bars: 168


Note what the purge costs: of 502 bars, only 168 count as evidence. Honest
evaluation is expensive by design — the alternative is evidence that isn't.

Across folds the platform also tracks **rank decay**: if config A beats config
B in fold 1 but loses in fold 3, the ranking is unstable and the sweep's
"winner" is partly luck. Rank decay is reported next to every sweep result.

## 5. PSR and DSR — why the platform doubts your Sharpe

A Sharpe ratio is a point estimate from a finite, non-normal sample. Two
corrections stand between it and a promotion
(`compute/stats/deflated_sharpe.py`, after Bailey & de Prado 2014):

- **PSR (Probabilistic Sharpe Ratio)** — the probability that the *true* Sharpe
  exceeds a benchmark, given how many OOS observations you have and how skewed
  and fat-tailed the returns are. A Sharpe of 1.5 over 60 bars of lucky,
  right-skewed returns can carry a lower PSR than a Sharpe of 0.9 over 1,300
  honest ones.
- **DSR (Deflated Sharpe Ratio)** — PSR measured against a *raised* benchmark
  `SR₀` that grows with the number of configurations you tried. If you test N
  strategies on random noise, the best one still looks great; SR₀ is the Sharpe
  the *luckiest of N noise strategies* would be expected to show:
  `SR₀ = std(SR across trials) × E[max of N standard normals]`.

Every configuration ever evaluated increments a **deflation clock that never
resets** — currently at 63 trials. Here is how fast the bar rises:

In [7]:
# Stdlib mirror of compute/stats/deflated_sharpe.py::_expected_max_sr_benchmark
# (Bailey & de Prado 2014). The platform's version uses scipy; NormalDist is identical.
import math
from statistics import NormalDist

GAMMA = 0.5772156649          # Euler–Mascheroni constant
Phi_inv = NormalDist().inv_cdf

def expected_max_of_n_normals(n_trials: int) -> float:
    if n_trials <= 1:
        return 0.0            # one trial -> no selection effect, benchmark stays 0
    return ((1 - GAMMA) * Phi_inv(1 - 1 / n_trials)
            + GAMMA * Phi_inv(1 - 1 / (n_trials * math.e)))

print("trials -> E[max] of that many lucky coin-flippers (in std-of-SR units)")
for n in (1, 6, 63):
    print(f"{n:>6} -> {expected_max_of_n_normals(n):.4f}")

trials -> E[max] of that many lucky coin-flippers (in std-of-SR units)
     1 -> 0.0000
     6 -> 1.3001
    63 -> 2.3635


After 63 trials, your best config must beat the luckiest of 63 coin-flippers —
about 2.36 standard deviations of cross-trial Sharpe spread — before its DSR
means anything. A strategy is only considered significant at **DSR > 0.95**.

### The scoreboard, with real numbers

From the platform's latest full-span evaluation
([`docs/research/full_span_verdicts.md`](research/full_span_verdicts.md)):
2020-01 to 2026-07 (~1,630 bars), anchored 4-fold WFO, `execution_lag_days=1`,
OOS n ≈ 1,300 bars per config:

| track | best config | OOS Sharpe | DSR | rank decay |
|---|---|---|---|---|
| PPO (reinforcement learning, 8-config grid) | lr 3e-4 | 0.756 | 0.898 | 0.32 |
| Supervised (6-config, ALPHA-1) | lgbm_21d | 0.905 | 0.943 | 0.33 |
| **Supervised + vol-target overlay (trial 63)** | **lgbm_21d + 10% vol target** | **1.134** | **0.921 (strict)** | — |

Nothing crosses DSR 0.95 — **no certified alpha yet, 63 trials on the clock**.

The vol-target row is the deflation clock earning its keep. The overlay lifted
the OOS Sharpe from 0.905 to 1.134 with all four folds positive, and a naive
benchmark counting only 3 trials printed DSR 0.988 — *certified!* — before the
strict computation, which deflates against all 63 trials in the honest-fill
pool, brought it back down to **0.921: not significant**. The platform rejected
its own most flattering number. Closest attempt so far: 0.92 vs the 0.95 bar.
The referee holds.

The other cautionary tale is in the history: an earlier evaluation on a short 2-year
window reported a PPO Sharpe of **2.17**. Extending the span and enforcing the
one-day execution lag collapsed it to **0.756** — the 2.17 was window luck plus
same-bar fill inflation, exactly the two failure modes this notebook has been
describing. The platform's verdict file calls those numbers "dead". A platform
that can't kill its own best result is an advertising machine, not a research
tool.

## 6. From backtest to paper orders

Suppose one day a config clears DSR 0.95. It still doesn't touch a broker
directly. The path (`research/promotion/`, `research/execution/`):

1. **Qualify** — a gate evaluation against a baseline: hard gates (block),
   soft gates (warn), execution-evidence gates (a replay under the production
   risk config), and regime gates.
2. **Promote** — tier by tier, `research → candidate → production`, order
   enforced; promotion to `production` requires an explicit
   `managing_partner_ack` that cannot be bypassed.
3. **Paper trade** — a long-running `PaperTradingLoop`
   (`research/execution/paper_trading.py`) sends orders to Alpaca's paper
   API with the full 15-constraint risk engine checking *before* each order
   and *after* each fill, a kill-switch state machine
   (`trading → halting → halted → resuming`) with `daily_loss`,
   `trailing_drawdown` and broker-failure triggers, a heartbeat table, and
   crash recovery that rebuilds positions from recorded fills and reconciles
   pending orders against the broker by ID.

Shadow, paper, and live runs all write to the same `execution_log` /
`execution_metrics` tables, distinguished by a `mode` column — so comparing
what a strategy *did* on paper with what its backtest *claimed* is one SQL
query. That comparison is the final honesty check.

## 7. Where to go next

- [`docs/ingestion.md`](ingestion.md) — bring your own symbols: the
  one-command backfill (`python -m yats_pipelines.backfill`), vendor
  transports, environment variables.
- [`research/scripts/`](../research/scripts/) — the WFO sweep runners
  (`wfo_sweep_2d.py`, `wfo_sweep_3d.py`, `wfo_sweep_4b.py`,
  `wfo_sweep_alpha1.py`) that produced every number above.
- [`docs/research/`](research/) — the sweep verdict files, written as
  research notes: what was tried, what survived, what died and why.
- [`docs/REFERENCE.md`](REFERENCE.md) — the full table and tool reference,
  including the MCP server that lets AI agents drive the platform.
- [`docs/yats_for_quanto_users.ipynb`](yats_for_quanto_users.ipynb) — a
  deeper tour of the pipeline (ingest → canonicalize → features → experiment →
  qualify → promote) with runnable job invocations.

If you take one thing from this notebook: when yats rejects your idea, it is
doing its job. It has already rejected 63 — including its own best one, twice.
The 64th trial is welcome to try.